In [1]:
import numpy as np
import pandas as pd
import glob
import os
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
def angle_distance_calculater(v_start, v_end):
    """ Returns the anti-clockwise angle in radians (0'-360') between 'v_end - v_start' and x axis (1,0) & the length of 'v_end - v_start'
    """
    vector = np.array(v_end) - np.array(v_start)
    angle_rad = np.arctan2(vector[1], vector[0])
    angle_deg = np.degrees(angle_rad)
    if angle_deg < 0:
        angle_deg += 360  # Ensure the angle is positive
    return angle_deg, np.linalg.norm(vector)


In [3]:
def get_theta_table(rating_data, valence_column, arousal_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, valence_sd_column, arousal_sd_column,cate_ori_rating_names, modality):
    start_stimulus = []
    end_stimulus = []
    group = []
    ori_start_category = []
    max_start_category = []
    ori_end_category = []
    max_end_category = []
    max_start_category_rank = []
    max_end_category_rank = []
    start_categories = {f'start_{category.split("_")[1]}': [] for category in category_columns}
    end_categories = {f'end_{category.split("_")[1]}': [] for category in category_columns}
    thetas = []
    location_on = []
    location_off = []
    distance = []
    start_valence = []
    start_valence_sd = []
    end_valence = []
    end_valence_sd = []
    start_arousal = []
    start_arousal_sd = []
    end_arousal = []
    end_arousal_sd = []
    on_grid = []
    off_grid = []
    for group_id in set(rating_data[group_column]):
        group_data = rating_data[rating_data[group_column]==group_id]
        for start_index in group_data.index:
            for end_index in group_data.index:
                if start_index != end_index:
                    theta_s_e, dist = angle_distance_calculater([group_data.loc[start_index,valence_column], group_data.loc[start_index,arousal_column]], 
                                    [group_data.loc[end_index,valence_column], group_data.loc[end_index,arousal_column]])
                    thetas.append(theta_s_e)

                    if 15 <= theta_s_e <= 45:
                        location_off.append(0)
                    elif 75 <= theta_s_e <= 105:
                        location_off.append(1)
                    elif 135 <= theta_s_e <= 165:
                        location_off.append(2)
                    elif 195 <= theta_s_e <= 225:
                        location_off.append(3)
                    elif 255 <= theta_s_e <= 285:
                        location_off.append(4)
                    elif 315 <= theta_s_e <= 345:
                        location_off.append(5)
                    else:
                        location_off.append(np.nan)
                    
                    if 0 <= theta_s_e <= 15:
                        location_on.append(0)
                    elif 345 <= theta_s_e <= 360:
                        location_on.append(0)
                    elif 45 <= theta_s_e <= 75:
                        location_on.append(1)
                    elif 105 <= theta_s_e <= 135:
                        location_on.append(2)
                    elif 165 <= theta_s_e <= 195:
                        location_on.append(3)
                    elif 225 <= theta_s_e <= 255:
                        location_on.append(4)
                    elif 285 <= theta_s_e <= 315:
                        location_on.append(5)
                    else:
                        location_on.append(np.nan)

                    #(theta_s_e+300/60) on
                    #theta_s_e/60 off

                    distance.append(dist)
                    on_grid.append(1 if (location_on[-1] != np.nan and location_off[-1] == np.nan) else 0)
                    off_grid.append(1 if (location_on[-1] == np.nan and location_off[-1] != np.nan) else 0)
                    start_stimulus.append(group_data.loc[start_index,stimulus_column])
                    end_stimulus.append(group_data.loc[end_index,stimulus_column])
                    group.append(group_id)
                    
                    ori_start_category.append(group_data.loc[start_index,ori_category_column])
                    max_start_category.append(group_data.loc[start_index,max_category_column])
                    ori_end_category.append(group_data.loc[end_index,ori_category_column])
                    max_end_category.append(group_data.loc[end_index,max_category_column])
                    current_start_max_category = group_data.loc[start_index,max_category_column] 
                    current_end_max_category = group_data.loc[end_index,max_category_column] 
                    #current_max_category is the value in cate_ori_rating_names, change to key
                    if pd.isna(current_start_max_category):
                        max_start_category_rank.append(np.nan)
                    else:
                        current_start_max_category = list(cate_ori_rating_names.keys())[list(cate_ori_rating_names.values()).index(current_start_max_category)] 
                        max_start_category_rank.append(group_data.loc[start_index,f'{modality}_{current_start_max_category}_slider.response_rank_mean'])
                    if pd.isna(current_end_max_category):
                        max_end_category_rank.append(np.nan)
                    else:
                        current_end_max_category = list(cate_ori_rating_names.keys())[list(cate_ori_rating_names.values()).index(current_end_max_category)] 
                        max_end_category_rank.append(group_data.loc[end_index,f'{modality}_{current_end_max_category}_slider.response_rank_mean'])

                    for category in category_columns:
                        start_categories[f'start_{category.split("_")[1]}'].append(group_data.loc[start_index, category])
                        end_categories[f'end_{category.split("_")[1]}'].append(group_data.loc[end_index, category])
                    start_valence.append(group_data.loc[start_index,valence_column])
                    start_arousal.append(group_data.loc[start_index,arousal_column])
                    end_valence.append(group_data.loc[end_index,valence_column])
                    end_arousal.append(group_data.loc[end_index,arousal_column])
                    start_valence_sd.append(group_data.loc[start_index,valence_sd_column])
                    start_arousal_sd.append(group_data.loc[start_index,arousal_sd_column])
                    end_valence_sd.append(group_data.loc[end_index,valence_sd_column])
                    end_arousal_sd.append(group_data.loc[end_index,arousal_sd_column])
    return pd.DataFrame({'group': group, 'start_stimulus': start_stimulus, 'end_stimulus': end_stimulus, 
                         'ori_start_category': ori_start_category, 'max_start_category': max_start_category,
                         'ori_end_category': ori_end_category, 'max_end_category': max_end_category, 'max_start_category_rank': max_start_category_rank, 'max_end_category_rank': max_end_category_rank} | start_categories | end_categories | {'theta': thetas, 'location_on': location_on, 'location_off': location_off, 'distance': distance, 
                            'start_valence': start_valence, 'start_arousal': start_arousal, 'end_valence': end_valence, 'end_arousal': end_arousal, 
                            'start_valence_sd': start_valence_sd, 'start_arousal_sd': start_arousal_sd, 'end_valence_sd': end_valence_sd, 'end_arousal_sd': end_arousal_sd,
                            'on_grid': on_grid, 'off_grid': off_grid})

def get_theta_table_mds(rating_data, mdsx_column, mdsy_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, cate_ori_rating_names, modality):
    start_stimulus = []
    end_stimulus = []
    group = []
    ori_start_category = []
    max_start_category = []
    ori_end_category = []
    max_end_category = []
    max_start_category_rank = []
    max_end_category_rank = []
    start_categories = {f'start_{category.split("_")[1]}': [] for category in category_columns}
    end_categories = {f'end_{category.split("_")[1]}': [] for category in category_columns}
    thetas = []

    distance = []
    start_mdsx = []
    start_mdsy = []
    end_mdsx = []
    end_mdsy = []
    for group_id in set(rating_data[group_column]):
        group_data = rating_data[rating_data[group_column]==group_id]
        for start_index in group_data.index:
            for end_index in group_data.index:
                if start_index != end_index:
                    theta_s_e, dist = angle_distance_calculater([group_data.loc[start_index,mdsx_column], group_data.loc[start_index,mdsy_column]], 
                                    [group_data.loc[end_index,mdsx_column], group_data.loc[end_index,mdsy_column]])
                    thetas.append(theta_s_e)
                    distance.append(dist)
                    start_stimulus.append(group_data.loc[start_index,stimulus_column])
                    end_stimulus.append(group_data.loc[end_index,stimulus_column])
                    group.append(group_id)
                    
                    ori_start_category.append(group_data.loc[start_index,ori_category_column])
                    max_start_category.append(group_data.loc[start_index,max_category_column])
                    ori_end_category.append(group_data.loc[end_index,ori_category_column])
                    max_end_category.append(group_data.loc[end_index,max_category_column])
                    current_start_max_category = group_data.loc[start_index,max_category_column] 
                    current_end_max_category = group_data.loc[end_index,max_category_column] 
                    #current_max_category is the value in cate_ori_rating_names, change to key
                    if pd.isna(current_start_max_category):
                        max_start_category_rank.append(np.nan)
                    else:
                        current_start_max_category = list(cate_ori_rating_names.keys())[list(cate_ori_rating_names.values()).index(current_start_max_category)] 
                        max_start_category_rank.append(group_data.loc[start_index,f'{modality}_{current_start_max_category}_slider.response_rank_mean'])
                    if pd.isna(current_end_max_category):
                        max_end_category_rank.append(np.nan)
                    else:
                        current_end_max_category = list(cate_ori_rating_names.keys())[list(cate_ori_rating_names.values()).index(current_end_max_category)] 
                        max_end_category_rank.append(group_data.loc[end_index,f'{modality}_{current_end_max_category}_slider.response_rank_mean'])

                    for category in category_columns:
                        start_categories[f'start_{category.split("_")[1]}'].append(group_data.loc[start_index, category])
                        end_categories[f'end_{category.split("_")[1]}'].append(group_data.loc[end_index, category])
                    start_mdsx.append(group_data.loc[start_index,mdsx_column])
                    start_mdsy.append(group_data.loc[start_index,mdsy_column])
                    end_mdsx.append(group_data.loc[end_index,mdsx_column])
                    end_mdsy.append(group_data.loc[end_index,mdsy_column])
    return pd.DataFrame({'group': group, 'start_stimulus': start_stimulus, 'end_stimulus': end_stimulus, 
                         'ori_start_category': ori_start_category, 'max_start_category': max_start_category,
                         'ori_end_category': ori_end_category, 'max_end_category': max_end_category, 'max_start_category_rank': max_start_category_rank, 'max_end_category_rank': max_end_category_rank} | start_categories | end_categories | {'theta': thetas, 'distance': distance, 
                            'start_mdsx': start_mdsx, 'start_mdsy': start_mdsy, 'end_mdsx': end_mdsx, 'end_mdsy': end_mdsy})

In [4]:
def get_normative_rating_data(allsubs_raw_rating_data_dir, valence_column, arousal_column, category_columns, ori_category_column, stimulus_column, group_column,cate_ori_rating_names, modality, sub_to_use='all', exclude_sub_from_all=None):
    allsubs_raw_rating_data_files = glob.glob(allsubs_raw_rating_data_dir+'/sub-*/*task-facewordRating*.csv')
    if exclude_sub_from_all and sub_to_use == 'all':
        allsubs_raw_rating_data_files = [file for file in allsubs_raw_rating_data_files if not any(exclude_sub in file for exclude_sub in exclude_sub_from_all)]
    if sub_to_use != 'all':
        allsubs_raw_rating_data_files = [file for file in allsubs_raw_rating_data_files if sub_to_use in file]
    allsubs_raw_rating_data = pd.concat([pd.read_csv(file, usecols=[valence_column, arousal_column]+category_columns+[ori_category_column, stimulus_column, group_column, 'participant']) for file in allsubs_raw_rating_data_files]).reset_index()
    if modality == 'word': #fill participant 0003's nan category_columns's values with 1
        mask = (allsubs_raw_rating_data['participant'] == 3) & (allsubs_raw_rating_data['word'].notna())
        allsubs_raw_rating_data.loc[mask, category_columns] = allsubs_raw_rating_data.loc[mask, category_columns].fillna(1)
    #fill in the group column for rows with values in face column and no value in group column (e.g., if face is ./stims/static_face/AM13DIS.JPG, then group is M13, so the 2nd to 4th characters)
    for index, row in allsubs_raw_rating_data.iterrows():
        if pd.isna(row[group_column]) and not pd.isna(row[stimulus_column]):
            allsubs_raw_rating_data.loc[index, group_column] = row[stimulus_column].split('/')[-1].split('.')[0][1:4]
            allsubs_raw_rating_data.loc[index, ori_category_column] = row['start_category']
    #omit na data
    #allsubs_raw_rating_data = allsubs_raw_rating_data.dropna()
    for stim in allsubs_raw_rating_data[stimulus_column].unique():
        stim_data = allsubs_raw_rating_data[allsubs_raw_rating_data[stimulus_column]==stim]
        for participant in stim_data['participant'].unique():
            participant_data = stim_data[stim_data['participant']==participant]
            category_data = participant_data[category_columns]
            category_ranks = category_data.T.rank(ascending=False, method='min')
            #write to allsubs_raw_rating_data and name the columns with _rank (not replace existing columns)
            for col in category_columns:
                allsubs_raw_rating_data.loc[(allsubs_raw_rating_data[stimulus_column]==stim) & (allsubs_raw_rating_data['participant']==participant), col+'_rank'] = category_ranks.T[col]

    #average across participants and get sd
    agg_dict = {valence_column: ['mean', 'std'], arousal_column: ['mean', 'std'], 'participant': 'count'}
    for col in category_columns:
        agg_dict[col] = ['mean', 'std']
    for col in category_columns:
        agg_dict[col+'_rank'] = ['mean']

    normative_rating_data = allsubs_raw_rating_data.groupby([group_column, stimulus_column, ori_category_column]).agg(agg_dict).reset_index()
    normative_rating_data.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in normative_rating_data.columns]
    normative_rating_data.rename(columns={'participant': f"{'participant'}_count"}, inplace=True)
    normative_rating_data.rename(columns={ori_category_column: "ori_category"}, inplace=True)

    category_columns = [col for col in normative_rating_data.columns if 'slider.response_mean' in col and f'{modality}_v_slider' not in col and f'{modality}_a_slider' not in col and 'rank' not in col]
    print(category_columns)
    normative_rating_data['max_category'] = normative_rating_data[category_columns].idxmax(axis=1)
    normative_rating_data['max_category'] = normative_rating_data['max_category'].str.replace('_slider.response_mean', '', regex=True).str.replace(f'{modality}_', '', regex=True)
    normative_rating_data['max_category'] = normative_rating_data['max_category'].map(cate_ori_rating_names)

    return normative_rating_data


### Valence-arousal ratings

In [ ]:
allsubs_raw_rating_data_dir = './data/beh/'
sub_to_use = 'all'#'sub-0004'#'all'#''#'all'#
exclude_sub_from_all = ['9999', '9998']
cate_ori_rating_names = {'hap': 'HAP', 'fear': 'AFR', 'sad': 'SAD', 'surprise': 'SUR', 'anger': 'ANG', 'neutral': 'NEU', 'disgust': 'DIS'}
all_data = {}
for modality in ['face', 'word']:
    valence_column = f'{modality}_v_slider.response'
    arousal_column = f'{modality}_a_slider.response'
    category_columns = [f'{modality}_hap_slider.response',f'{modality}_fear_slider.response',f'{modality}_sad_slider.response',f'{modality}_surprise_slider.response',f'{modality}_anger_slider.response',f'{modality}_neutral_slider.response',f'{modality}_disgust_slider.response']
    ori_category_column = 'category' if modality == 'word' else 'start_category'
    stimulus_column = modality
    group_column = 'group'
    
    
    norm_data = get_normative_rating_data(allsubs_raw_rating_data_dir, valence_column, arousal_column, category_columns, ori_category_column, stimulus_column, group_column, cate_ori_rating_names, modality, sub_to_use, exclude_sub_from_all)
    all_data[modality] = norm_data
    #get min and max of each std column separately for each group
    for group in set(norm_data[group_column]):
        for col in [valence_column, arousal_column]+category_columns:
            print(group, 'min', col, norm_data[norm_data[group_column]==group][col+'_std'].min())
            print(group,'max', col, norm_data[norm_data[group_column]==group][col+'_std'].max())
            print(group, 'mean', col, norm_data[norm_data[group_column]==group][col+'_std'].mean())
        print('')
    #save word and face data to csv
#    if sub_to_use == 'all':
#        norm_data.to_csv(f'./data/beh/behTables_singleStim/subAvg_table_{modality}_singleStim.csv', index=False)
#    else:
#        sub_id_str = sub_to_use.replace('-', '')
#        if not os.path.exists(f'./data/beh/behTables_singleStim/{sub_id_str}'):
#            os.makedirs(f'./data/beh/behTables_singleStim/{sub_id_str}')
#        norm_data.to_csv(f'./data/beh/behTables_singleStim/{sub_id_str}/{sub_id_str}_table_{modality}_singleStim.csv', index=False)

In [8]:
source_type = 'subAvg' if sub_to_use == 'all' else sub_to_use
cate_ori_rating_names = {'hap': 'HAP', 'fear': 'AFR', 'sad': 'SAD', 'surprise': 'SUR', 'anger': 'ANG', 'neutral': 'NEU', 'disgust': 'DIS'}
for modality in ['word', 'face']:   
    valence_column = f'{modality}_v_slider.response_mean'
    valence_sd_column = f'{modality}_v_slider.response_std'
    arousal_column = f'{modality}_a_slider.response_mean'
    arousal_sd_column = f'{modality}_a_slider.response_std' 
    ori_category_column = 'ori_category' 
    max_category_column = 'max_category'
    stimulus_column = 'word' if modality == 'word' else 'face'
    group_column = 'group'
    category_columns = [f'{modality}_hap_slider.response_mean',f'{modality}_fear_slider.response_mean',f'{modality}_sad_slider.response_mean',f'{modality}_surprise_slider.response_mean',f'{modality}_anger_slider.response_mean',f'{modality}_neutral_slider.response_mean',f'{modality}_disgust_slider.response_mean']
    if source_type == 'subAvg':
        get_theta_table(all_data[modality], valence_column, arousal_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, valence_sd_column, arousal_sd_column,cate_ori_rating_names, modality).to_csv(f'./data/beh/behTables/{source_type}_table_{modality}.csv', index=False)
    else:
        source_type = source_type.replace('-', '')
        if not os.path.exists(f'./data/beh/behTables/{source_type}'):
            os.makedirs(f'./data/beh/behTables/{source_type}')
        get_theta_table(all_data[modality], valence_column, arousal_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, valence_sd_column, arousal_sd_column,cate_ori_rating_names, modality).to_csv(f'./data/beh/behTables/{source_type}/{source_type}_table_{modality}.csv', index=False)

### MDS

In [5]:
def add_mds_coords(rating_df, categories, modality, metric='cosine', random_state=42):
    rating_cols = [f'{modality}_{cat}_slider.response_mean' for cat in categories]

    #drop rows with any nan in rating columns
    no_nan_df = rating_df.dropna(subset=rating_cols).copy()
    X = no_nan_df[rating_cols].to_numpy(dtype=float)

    #zero-variance rows (can cause nans for correlation/cosine)
    if metric in ['correlation', 'cosine']:
        stds = X.std(axis=1)
        nonzero_var = stds > 1e-8
        if not np.all(nonzero_var):
            #print(f"Dropping {np.sum(~nonzero_var)} zero-variance rows for metric='{metric}'")
            dropped_zero_var_df = no_nan_df.loc[~nonzero_var].copy()
            no_nan_df = no_nan_df.loc[nonzero_var].copy()
            X = X[nonzero_var]
            #print(dropped_zero_var_df)
            
    dissimilarity = pairwise_distances(X, metric=metric)
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=random_state)
    coords = mds.fit_transform(dissimilarity)
    no_nan_df['mds_x'] = coords[:, 0]
    no_nan_df['mds_y'] = coords[:, 1]
    return no_nan_df

In [ ]:
## Get theta table from MDS for 10 seeds
subjects = ['0001', '0002', '0003', '0004', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0015', '0016', '0017', '0018', '0019', '0021']
categories = ['hap', 'fear', 'sad', 'surprise', 'anger','neutral','disgust']
subavg_dfs = {'face':pd.read_csv('./data/beh/behTables_singleStim/subAvg_table_face_singleStim.csv'),
              'word':pd.read_csv('./data/beh/behTables_singleStim/subAvg_table_word_singleStim.csv')}
allsubs_dfs = {}
for sub in subjects:
    allsubs_dfs[sub] = {}
    for modality in ['face', 'word']:
        allsubs_dfs[sub][modality] = pd.read_csv(f'./data/beh/behTables_singleStim/sub{sub}/sub{sub}_table_{modality}_singleStim.csv')


cate_ori_rating_names = {'hap': 'HAP', 'fear': 'AFR', 'sad': 'SAD', 'surprise': 'SUR', 'anger': 'ANG', 'neutral': 'NEU', 'disgust': 'DIS'}
for seed in [42]+list(range(1, 11)):
    for source_type in ['subAvg', 'subspec']:
        for modality in ['word', 'face']:   
            for metric in ['cosine', 'correlation']:
                mdsx_column = 'mds_x'
                mdsy_column = 'mds_y'
                ori_category_column = 'ori_category' 
                max_category_column = 'max_category'
                stimulus_column = 'word' if modality == 'word' else 'face'
                group_column = 'group'
                category_columns = [f'{modality}_hap_slider.response_mean',f'{modality}_fear_slider.response_mean',f'{modality}_sad_slider.response_mean',f'{modality}_surprise_slider.response_mean',f'{modality}_anger_slider.response_mean',f'{modality}_neutral_slider.response_mean',f'{modality}_disgust_slider.response_mean']
                if source_type == 'subAvg':
                    get_theta_table_mds(add_mds_coords(subavg_dfs[modality], categories, modality, metric=metric, random_state=seed), mdsx_column, mdsy_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, cate_ori_rating_names, modality).to_csv(f'./data/beh/behTables/MDS/{source_type}_table_{modality}_mds{metric}{seed}.csv', index=False)
                else:
                    for sub in subjects:
                        source_type = f'sub{sub}'
                        print(source_type)
                        if not os.path.exists(f'./data/beh/behTables/MDS/{source_type}'):
                            os.makedirs(f'./data/beh/behTables/MDS/{source_type}')
                        get_theta_table_mds(add_mds_coords(allsubs_dfs[sub][modality], categories, modality, metric=metric, random_state=seed), mdsx_column, mdsy_column, category_columns, ori_category_column, max_category_column, stimulus_column, group_column, cate_ori_rating_names, modality).to_csv(f'./data/beh/behTables/MDS/{source_type}/{source_type}_table_{modality}_mds{metric}{seed}.csv', index=False)


In [ ]:
from matplotlib.lines import Line2D

def plot_aligned_overlay(ref_df, aligned_df, modality, metric, source_type, seed, outdir='./data/beh/behTables/MDS/plots'):
    sns.set(style="white", context="talk")

    ref_coords = ref_df[['mds_x', 'mds_y']].to_numpy()
    aligned_coords = aligned_df[['mds_x', 'mds_y']].to_numpy()
    labels = ref_df['max_category'].values if 'max_category' in ref_df.columns else None

    plt.figure(figsize=(6,6))
    ax = plt.gca()

    # connecting lines
    for i in range(len(ref_coords)):
        ax.plot([ref_coords[i,0], aligned_coords[i,0]],
                [ref_coords[i,1], aligned_coords[i,1]],
                color='gray', alpha=0.5, linewidth=0.8, zorder=1)

    # reference points
    sns.scatterplot(x=ref_coords[:,0], y=ref_coords[:,1],
        hue=labels if labels is not None else None,
        palette='husl' if labels is not None else None,
        s=70, edgecolor='black', linewidth=0.7,
        alpha=0.9, zorder=3, legend=(labels is not None))

    # aligned points (red outlines)
    ax.scatter(aligned_coords[:,0], aligned_coords[:,1],s=60, facecolor='none', edgecolor='red', linewidth=1.2,zorder=4)

    # legend 
    handles, legend_labels = ax.get_legend_handles_labels()
    custom = [
        Line2D([0], [0], marker='o', color='black', label='reference',
               markerfacecolor='gray', markersize=6, linewidth=0),
        Line2D([0], [0], marker='o', color='red', label=f'aligned seed {seed}',
               markerfacecolor='none', markersize=6, linewidth=1.2)
    ]
    ax.legend(handles=custom + handles if labels is not None else custom,
              bbox_to_anchor=(1.05,1), loc='upper left', fontsize='small')

    ax.set_title(f"{source_type} – {modality} – {metric}\nSeed 42 (ref) vs Seed {seed} (aligned)")
    ax.set_xlabel("MDS-1 (aligned)")
    ax.set_ylabel("MDS-2 (aligned)")
    ax.set_aspect('equal', 'box')
    sns.despine()
    plt.tight_layout()
   # plt.show()


def circular_correlation(theta1, theta2):
    theta1 = np.asarray(theta1)
    theta2 = np.asarray(theta2)
    # remove NaNs
    mask = ~np.isnan(theta1) & ~np.isnan(theta2)
    theta1, theta2 = theta1[mask], theta2[mask]

    if len(theta1) == 0:
        return np.nan

    sin1 = np.sin(theta1 - np.mean(theta1))
    sin2 = np.sin(theta2 - np.mean(theta2))
    r = np.sum(sin1 * sin2) / np.sqrt(np.sum(sin1 ** 2) * np.sum(sin2 ** 2))
    return r



In [ ]:
## Align MDS coordinates to the first seed and get average theta across seeds
from scipy.spatial import procrustes
from itertools import combinations
from scipy.stats import circmean

subjects = ['0001', '0002', '0003', '0004', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0015', '0016', '0017', '0018', '0019', '0021']
categories = ['hap', 'fear', 'sad', 'surprise', 'anger','neutral','disgust']
subavg_dfs = {'face':pd.read_csv('./data/beh/behTables_singleStim/subAvg_table_face_singleStim.csv'),
              'word':pd.read_csv('./data/beh/behTables_singleStim/subAvg_table_word_singleStim.csv')}
allsubs_dfs = {sub: {mod: pd.read_csv(f'./data/beh/behTables_singleStim/sub{sub}/sub{sub}_table_{mod}_singleStim.csv')
                     for mod in ['face','word']} for sub in subjects}

cate_ori_rating_names = {'hap': 'HAP', 'fear': 'AFR', 'sad': 'SAD', 'surprise': 'SUR', 'anger': 'ANG', 'neutral': 'NEU', 'disgust': 'DIS'}

seeds = list(range(1, 11))
circular_corrs_all = []
disparity_all = []
for source_type in ['subAvg', 'subspec']:
    for modality in ['word', 'face']:
        for metric in ['cosine', 'correlation']:

            if source_type == 'subAvg':
                ref_df = add_mds_coords(subavg_dfs[modality], categories, modality,metric=metric, random_state=seeds[0])
                ref_coords = ref_df[['mds_x','mds_y']].to_numpy()
                theta_tables = {}
                theta_table_ref = get_theta_table_mds(ref_df,'mds_x','mds_y',[f'{modality}_{cat}_slider.response_mean' for cat in categories],'ori_category','max_category','word' if modality=='word' else 'face','group',cate_ori_rating_names,modality)
                outdir = './data/beh/behTables/MDS_aligned'
                os.makedirs(outdir, exist_ok=True)
                theta_table_ref.to_csv(f'{outdir}/{source_type}_table_{modality}_mds{metric}{seeds[0]}.csv',index=False)
                theta_tables[seeds[0]] = theta_table_ref

                # align other seeds
                for seed in seeds[1:]:
                    curr_df = add_mds_coords(subavg_dfs[modality], categories, modality,metric=metric, random_state=seed)
                    curr_coords = curr_df[['mds_x','mds_y']].to_numpy()
                    mtx1, mtx2, disparity = procrustes(ref_coords, curr_coords)
                    curr_df['mds_x'], curr_df['mds_y'] = mtx2[:,0], mtx2[:,1]
                    disparity_all.append({'source_type':source_type,'modality':modality,'metric':metric,'seed':seed,'disparity':disparity, 'sub': 'subAvg'})

                    #plot_aligned_overlay(ref_df, curr_df, modality, metric, source_type, seed)

                    theta_table = get_theta_table_mds(
                        curr_df,'mds_x','mds_y',
                        [f'{modality}_{cat}_slider.response_mean' for cat in categories],
                        'ori_category','max_category',
                        'word' if modality=='word' else 'face','group',
                        cate_ori_rating_names,modality
                    )
                    theta_table.to_csv(f'{outdir}/{source_type}_table_{modality}_mds{metric}{seed}.csv',index=False)
                    theta_tables[seed] = theta_table
                
                #get average theta across seeds
                theta_table_avg = theta_tables[seeds[0]].copy()
                # compute average theta vector across all seeds
                thetas = np.stack([np.deg2rad(theta_tables[seed]['theta'].values) for seed in seeds])
                theta_table_avg['theta'] = np.rad2deg(circmean(thetas, axis=0))
                theta_table_avg.to_csv(f'{outdir}/{source_type}_table_{modality}_mds{metric}seedAvg.csv',index=False)

                # circular correlations
                for s1, s2 in combinations(seeds,2):
                    th1 = np.array(theta_tables[s1]['theta'])
                    th2 = np.array(theta_tables[s2]['theta'])
                    r = circular_correlation(th1, th2)
                    circular_corrs_all.append({'source_type':source_type,'modality':modality,'metric':metric,'seed1':s1,'seed2':s2,'r_circ':r})

            else:
                for sub in subjects:
                    sub_label = f'sub{sub}'
                    sub_outdir = f'./data/beh/behTables/MDS_aligned/{sub_label}'
                    os.makedirs(sub_outdir, exist_ok=True)

                    ref_df = add_mds_coords(allsubs_dfs[sub][modality], categories, modality,metric=metric, random_state=seeds[0])
                    ref_coords = ref_df[['mds_x','mds_y']].to_numpy()
                    theta_tables = {}
                    theta_table_ref = get_theta_table_mds(
                        ref_df,'mds_x','mds_y',
                        [f'{modality}_{cat}_slider.response_mean' for cat in categories],
                        'ori_category','max_category',
                        'word' if modality=='word' else 'face','group',
                        cate_ori_rating_names,modality
                    )
                    theta_table_ref.to_csv(f'{sub_outdir}/{sub_label}_table_{modality}_mds{metric}{seeds[0]}.csv',index=False)
                    theta_tables[seeds[0]] = theta_table_ref

                    for seed in seeds[1:]:
                        curr_df = add_mds_coords(allsubs_dfs[sub][modality], categories, modality,metric=metric, random_state=seed)
                        curr_coords = curr_df[['mds_x','mds_y']].to_numpy()
                        mtx1, mtx2, disparity = procrustes(ref_coords, curr_coords)
                        curr_df['mds_x'], curr_df['mds_y'] = mtx2[:,0], mtx2[:,1]
                        disparity_all.append({'source_type':source_type,'modality':modality,'metric':metric,'seed':seed,'disparity':disparity, 'sub': sub_label})

                        theta_table = get_theta_table_mds(curr_df,'mds_x','mds_y',[f'{modality}_{cat}_slider.response_mean' for cat in categories],'ori_category','max_category','word' if modality=='word' else 'face','group',cate_ori_rating_names,modality)
                        theta_table.to_csv(f'{sub_outdir}/{sub_label}_table_{modality}_mds{metric}{seed}.csv',index=False)
                        theta_tables[seed] = theta_table
                    #get average theta across seeds
                    theta_table_avg = theta_tables[seeds[0]].copy()
                    thetas = np.stack([np.deg2rad(theta_tables[seed]['theta'].values) for seed in seeds])
                    theta_table_avg['theta'] = np.rad2deg(circmean(thetas, axis=0))
                    theta_table_avg.to_csv(f'{sub_outdir}/{sub_label}_table_{modality}_mds{metric}seedAvg.csv',index=False)

                    # circular correlations per subject
                    for s1, s2 in combinations(seeds,2):
                        th1 = np.deg2rad(theta_tables[s1]['theta'])
                        th2 = np.deg2rad(theta_tables[s2]['theta'])
                        r = circular_correlation(th1, th2)
                        circular_corrs_all.append({'source_type':source_type,'subject':sub,'modality':modality,'metric':metric,'seed1':s1,'seed2':s2,'r_circ':r})

In [ ]:
circ_corr_df = pd.DataFrame(circular_corrs_all)
print(circ_corr_df.groupby(['source_type', 'modality', 'metric'])['r_circ'].mean())
print(pd.DataFrame(disparity_all).to_string(max_rows=None, max_cols=None))